# Student C - Charts and Management Report

Name: Rakibul Hussain
Role: Student C


Responsibilities:
- Create 3 Matplotlib charts (trend, comparison, relationship).
- Calculate 3 numerical findings from the supplied tables.
- Link each finding to a practical action.
- State coverage, the documented anomalies and two limitations.
- Write the plain-text management report.

In [1]:
from pathlib import Path
import importlib.util
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / '02_data.csv').exists(), 'Launch Jupyter from the project folder'
def load(filename):
    spec=importlib.util.spec_from_file_location(Path(filename).stem, ROOT/filename)
    mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod
RUN_MY_CHECKS = True
checks=load('11_checks.py')


### Implementation Decisions

1. The comparison bar chart starts at 0%. A truncated axis made a 2.7-point gap between buildings look huge.

2. Every finding is calculated inside `create_report` from the `daily`, `entities` and `detail` tables; nothing is hard-coded. The missing key (E06 / July 1) is found by comparing expected and observed date + entity pairs.

In [2]:
raw=pd.read_csv(ROOT/'02_data.csv',dtype=str,keep_default_na=False)
print('Raw rows:',len(raw))
print(raw.head().to_string(index=False))

Raw rows: 726
      date entity_id segment generation_kwh load_kwh export_kwh import_kwh
2026-08-05       E05 Central            223      319          0         96
2026-07-16       E12   South            309      383          0         74
2026-08-12       E09   South            266      361          0         95
2026-07-16       E02   North            247      221         26          0
2026-08-15       E07 Central            201      320          0        119


Raw rows: 726
      date entity_id segment generation_kwh load_kwh export_kwh import_kwh
2026-08-05       E05 Central            223      319          0         96
2026-07-16       E12   South            309      383          0         74
2026-08-12       E09   South            266      361          0         95
2026-07-16       E02   North            247      221         26          0
2026-08-15       E07 Central            201      320          0        119

### Bug Fixed

Problem:
In the first trend chart the 'Lowest' label overlapped the x-axis dates and the legend.

Solution:
I added y-axis padding with `set_ylim`, moved the legend to the lower left and offset the annotation labels.

In [3]:
mine=load('09_student_C.py')
if RUN_MY_CHECKS:
    a=load('07_student_A.py');b=load('08_student_B.py')
    clean,audit=a.load_and_clean(ROOT/'02_data.csv')
    detail,daily,entities=b.calculate_metrics(clean)
    paths=mine.create_report(detail,daily,entities,ROOT/'results')
    checks.check_c(paths)
    print([str(p) for p in paths])
else:
    print('TODO: implementation checks have NOT run.')

['D:\\ngf project 4\\results\\01_trend.png', 'D:\\ngf project 4\\results\\02_comparison.png', 'D:\\ngf project 4\\results\\03_relationship.png', 'D:\\ngf project 4\\results\\04_report.txt']


mine=load('09_student_C.py')
if RUN_MY_CHECKS:
    a=load('07_student_A.py');b=load('08_student_B.py')
    clean,audit=a.load_and_clean(ROOT/'02_data.csv')
    detail,daily,entities=b.calculate_metrics(clean)
    paths=mine.create_report(detail,daily,entities,ROOT/'results')
    checks.check_c(paths)
    print([str(p) for p in paths])
else:
    print('TODO: implementation checks have NOT run.')

### Added edge-case test

A zero-activity row has an undefined rate (NaN). The charts and report must not crash on it, and the NaN must not be replaced by zero.

In [4]:
import numpy as np, tempfile
toy_detail = pd.DataFrame({'date': pd.to_datetime(['2026-07-01','2026-07-01']),
    'entity_id': ['E01','E02'], 'generation_kwh': [100.0, 0.0], 'load_kwh': [90.0, 0.0],
    'export_kwh': [20.0, 0.0], 'import_kwh': [10.0, 0.0],
    'rate_numerator': [80.0, 0.0], 'rate_denominator': [100.0, 0.0],
    'self_consumption_pct': [80.0, np.nan]})
toy_daily = pd.DataFrame({'date': pd.to_datetime(['2026-07-01','2026-07-02']),
    'generation_kwh': [100.0, 0.0], 'load_kwh': [90.0, 0.0], 'export_kwh': [20.0, 0.0],
    'import_kwh': [10.0, 0.0], 'rate_numerator': [80.0, 0.0], 'rate_denominator': [100.0, 0.0],
    'weighted_self_consumption_pct': [80.0, np.nan]})
toy_ent = pd.DataFrame({'entity_id': ['E01','E02'], 'generation_kwh': [100.0, 0.0],
    'load_kwh': [90.0, 0.0], 'export_kwh': [20.0, 0.0], 'import_kwh': [10.0, 0.0],
    'rate_numerator': [80.0, 0.0], 'rate_denominator': [100.0, 0.0],
    'weighted_self_consumption_pct': [80.0, np.nan]})
with tempfile.TemporaryDirectory() as td:
    out = mine.create_report(toy_detail, toy_daily, toy_ent, td)
    assert len(out) == 4 and all(p.exists() for p in out)
print('Edge-case test passed (toy output discarded, not used as findings).')

Edge-case test passed (toy output discarded, not used as findings).


Edge-case test passed (toy output discarded, not used as findings).

In [5]:
print('Detail rows:', len(detail), '| Daily rows:', len(daily), '| Entity rows:', len(entities))
print('Files created:', [p.name for p in paths])
print()
print((ROOT/'results'/'04_report.txt').read_text(encoding='utf-8'))

Detail rows: 719 | Daily rows: 60 | Entity rows: 12
Files created: ['01_trend.png', '02_comparison.png', '03_relationship.png', '04_report.txt']

SOLAR ENERGY SELF-CONSUMPTION - MANAGEMENT REPORT (synthetic teaching data)

Period: July 1 - August 29, 2026. Buildings: 12.
Coverage: 719 of 720 expected building-days (719/720). Missing key: E06 / July 1 (conflicting records were quarantined).
Headline: Most solar output is used locally, but the gap between best and worst buildings (2.7 points) is worth a closer look.

THREE FINDINGS (calculated from the supplied tables)
1. Overall weighted self-consumption is 85.32% of generation. Buildings range from 83.76% (E06) to 86.49% (E04), a spread of 2.73 points.
2. The daily weighted rate is lowest on August 22 (74.74%) and highest on July 4 (95.83%).
3. Grid dependence (import / load) is 19.9% overall. It is highest for E08 at 27.6% and on August 12 at 34.9% across all buildings.

THREE FEASIBLE ACTIONS
1. Review E06 (lowest rate) and compare i

Detail rows: 719 | Daily rows: 60 | Entity rows: 12
Files created: ['01_trend.png', '02_comparison.png', '03_relationship.png', '04_report.txt']

SOLAR ENERGY SELF-CONSUMPTION - MANAGEMENT REPORT (synthetic teaching data)
========================================================================

Period: July 1 - August 29, 2026. Buildings: 12.
Coverage: 719 of 720 expected building-days (719/720). Missing key: E06 / July 1 (conflicting records were quarantined).
Headline: Most solar output is used locally, but the gap between best and worst buildings (2.7 points) is worth a closer look.

THREE FINDINGS (calculated from the supplied tables)
1. Overall weighted self-consumption is 85.32% of generation. Buildings range from 83.76% (E06) to 86.49% (E04), a spread of 2.73 points.
2. The daily weighted rate is lowest on August 22 (74.74%) and highest on July 4 (95.83%).
3. Grid dependence (import / load) is 19.9% overall. It is highest for E08 at 27.6% and on August 12 at 34.9% across all buildings.

THREE FEASIBLE ACTIONS
1. Review E06 (lowest rate) and compare its export pattern with E04 (highest) to see what differs in daytime use.
2. Check what happened around August 22 (lowest daily rate) using more detailed data before deciding on any change.
3. Investigate why E08 imports the largest share of its load and whether shifting flexible loads to daylight hours is practical.

DOCUMENTED CASES
E01 on July 21 had generation of 141 kWh and a row rate of 100.0%. It is a documented low-generation day and is retained. We do not attribute any change in the daily total to it.
Zero-activity closure row retained: E07 on August 10. Its rate is undefined (NaN), not 0%, and it adds nothing to weighted totals.

IS A HIGHER RATE DESIRABLE?
Generally yes here: a higher rate means more of the generated solar energy was used on site instead of exported. But the rate only describes generation. A building can be at 100% and still import a lot of grid power, so read it together with import / load.

TWO LIMITATIONS
1. Daily netting hides within-day imports and exports. These simplified meters cannot support tariff savings or battery sizing.
2. Data are synthetic and 719/720 coverage has one missing key. No causal claim or real-world impact can be made from these results; the weighted rate is a daily rate, not a measurement of cost or savings.

## Explain one result

The overall weighted self-consumption rate is 85.32%. It is total self-consumed energy divided by total generation, not an average of daily percentages, so high-generation days count more. A higher rate means more solar was used on site, but it does not prove savings: daily netting hides within-day imports and exports, and the data are synthetic.

**Submission:** Save this executed notebook and `09_student_C.py`, and provide the 3 PNGs and the report with the group project.

INDIVIDUAL SUBMISSION CHECKLIST

Name, two decisions, one bug fixed and one new edge-case test are added. Checks are enabled. Restart and run all cells before submitting.